# 제조 AI 심화과정 — 실습② 내 손으로 만드는 불량 검출기 (완성본 v1.2)

**삼성전자 × KAMPA · 1일차 오후 14:15–15:50 (95분) · ㈜에이비에이치**

- 데이터: 실제 주조 공장 임펠러 사진 **7,348장** (캐글 공개 데이터셋)
- 목표: ① 전이학습으로 양불 분류기 제작 → ② Grad-CAM으로 판정 근거 확인
- 환경: Google Colab + **T4 GPU** (설치 불필요)
- 진행: 위에서부터 **한 셀씩** 실행. `[체크]` 표시 값이 나오면 다음으로.

> ⚠️ 시작 전 필수: 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장**
> 저장하면 세션이 초기화되어 변수가 사라집니다 — **정상 동작**이니 놀라지 마세요. (강의안 28p)

---
## STEP 0 · GPU 연결 확인 (28p)

In [ ]:
# [체크] 아래 실행 결과에 GPU:0 한 줄이 보이면 통과 → 공유시트 체크 ①
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(gpus)
assert len(gpus) > 0, "GPU가 안 보입니다! 런타임 → 런타임 유형 변경 → T4 GPU → 저장 후 다시 실행"
print("\n✅ GPU 연결 정상 — 다음 셀로 진행하세요.")

In [ ]:
# 한글 그림 폰트 설치·설정 (약 10초) — 그림 제목이 □□로 깨지는 것 방지
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print("✅ 한글 폰트 설정 완료 — 그림의 한글이 정상 표시됩니다.")

---
## STEP 1 · 데이터 다운로드 — Kagglehub (29p)
캐글의 실제 주조(Casting) 공장 데이터셋을 내려받습니다. 1~2분 소요.

In [ ]:
%pip install -q kagglehub
# 이미 설치되어 있으면 수 초 내 통과합니다.

In [ ]:
# 주조 임펠러 데이터셋 다운로드 (공개 데이터셋 — 로그인 불필요)
import kagglehub, os

path = kagglehub.dataset_download("ravirajsinh45/real-life-industrial-dataset-of-casting-product")
print("Dataset path:", path)

# 폴더 구조 자동 탐색 (train/test 아래 def_front·ok_front가 있는 300x300 세트)
train_dir = test_dir = None
for root, dirs, files in os.walk(path):
    if "def_front" in dirs and "ok_front" in dirs:
        if os.path.basename(root) == "train":
            train_dir = root
        elif os.path.basename(root) == "test":
            test_dir = root
assert train_dir and test_dir, "train/test 폴더를 찾지 못했습니다 — 아래 백업 셀로 우회하세요."

def count(d):
    return {c: len(os.listdir(os.path.join(d, c))) for c in ["def_front", "ok_front"]}

tr, te = count(train_dir), count(test_dir)
total = sum(tr.values()) + sum(te.values())
print(f"train  불량(def) {tr['def_front']:,} / 정상(ok) {tr['ok_front']:,}")
print(f"test   불량(def) {te['def_front']:,} / 정상(ok) {te['ok_front']:,}")
print(f"총 {total:,}장")
# [체크] train 3,758/2,875 · test 453/262 · 총 7,348장이면 통과 → 공유시트 체크 ②
assert total == 7348, "장수가 다릅니다 — 다운로드가 불완전할 수 있으니 백업 셀로 우회하세요."
print("\n✅ 데이터 구조 검증 완료")

In [ ]:
# ── (백업) 다운로드가 트래픽/네트워크로 실패할 때만 이 셀의 주석(#)을 풀고 실행 ──
# 강사 제공 구글 드라이브 고정 주소로 우회합니다. (강의안 29p 하단 안내)
# FILE_ID = "「8/25 백업 zip 업로드 후 기입」"
# !gdown --id $FILE_ID -O casting_backup.zip
# !unzip -qo casting_backup.zip -d casting_backup
# import os
# train_dir = "casting_backup/casting_data/casting_data/train"
# test_dir  = "casting_backup/casting_data/casting_data/test"
# print(os.listdir(train_dir), os.listdir(test_dir))

---
## STEP 2 · 육안 검사 & 라벨 트랩 (30p)
모델에 넣기 전에 **사람 눈**부터. 기공·버·모서리 결손을 직접 찾아보세요 (30초).

In [ ]:
# 3×3 그리드 — 불량/정상 무작위 9장 육안 검사
import random, matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img

samples = []
for cls, tag in [("def_front", "불량"), ("ok_front", "정상")]:
    files = os.listdir(os.path.join(train_dir, cls))
    for f in random.sample(files, 5 if cls == "def_front" else 4):
        samples.append((os.path.join(train_dir, cls, f), tag))
random.shuffle(samples)

plt.figure(figsize=(9, 9))
for i, (fp, tag) in enumerate(samples[:9]):
    plt.subplot(3, 3, i + 1)
    plt.imshow(load_img(fp), cmap="gray"); plt.axis("off"); plt.title(tag, fontsize=11)
plt.suptitle("주조 임펠러 육안 검사 — 어디가 결함일까요?", y=0.93)
plt.show()

In [ ]:
# 데이터셋 로드 + ⚠️ 오늘의 최다 오답 지점: 라벨 트랩
from tensorflow.keras.utils import image_dataset_from_directory

IMG, BATCH = (224, 224), 32
train_ds_raw = image_dataset_from_directory(train_dir, validation_split=0.2, subset="training",
                                            seed=42, image_size=IMG, batch_size=BATCH)
val_ds_raw   = image_dataset_from_directory(train_dir, validation_split=0.2, subset="validation",
                                            seed=42, image_size=IMG, batch_size=BATCH)
class_names = train_ds_raw.class_names
print("class_names:", class_names)   # 알파벳 순 정렬!

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.prefetch(AUTOTUNE)
val_ds   = val_ds_raw.prefetch(AUTOTUNE)

print()
print("=" * 46)
print("  라벨 트랩 (강의안 30p) — 폴더 알파벳 순서 때문에")
print("  def_front(불량) = 0  /  ok_front(정상) = 1")
print("  → 시그모이드 출력 0.5 초과 = '정상' 판정")
print("  직관과 반대! 페어와 소리 내어 확인: \"불량이 0, 정상이 1\"")
print("=" * 46)
assert class_names == ["def_front", "ok_front"]

---
## STEP 3 · 데이터 증강 미리보기 (31p)
**금기 규칙** — 현실에서 일어날 수 없는 변형은 독(과적합·오탐):
- ① 문자 각인 부품 → 좌우 반전(Flip) 금지
- ② 조립 방향 고정 부품 → 무작위 회전(Rotation) 금지

오늘의 임펠러는 **원형 대칭 + 상면 고정 촬영**이라 회전·반전 모두 안전합니다.
(아래는 개념 확인용 시각화 — 본 실습 학습에는 적용하지 않고 강의안 32p 코드 그대로 갑니다.)

In [ ]:
# 안전한 증강 예시 시각화 — 원본 1장 → 회전·반전 4변형
from tensorflow.keras import layers
import numpy as np

aug = tf.keras.Sequential([layers.RandomFlip("horizontal_and_vertical"), layers.RandomRotation(0.3)])
img = np.array(load_img(samples[0][0], target_size=IMG), dtype="float32")[None, ...]

plt.figure(figsize=(11, 2.6))
plt.subplot(1, 5, 1); plt.imshow(img[0].astype("uint8")); plt.axis("off"); plt.title("원본")
for i in range(4):
    out = aug(img, training=True)
    plt.subplot(1, 5, i + 2); plt.imshow(out[0].numpy().astype("uint8")); plt.axis("off"); plt.title(f"증강 {i+1}")
plt.suptitle("원형 대칭 부품 — 물리적으로 허용되는 증강", y=1.05)
plt.show()

---
## STEP 4 · MobileNetV2 전이학습 모델 (32p)
**눈은 빌리고, 기준만 가르친다** — 백본 동결(약 224만 개) + 판정 헤드(1,281개)만 학습.

In [ ]:
# 1. MobileNetV2 백본 로드 및 동결 (강의안 32p 코드와 동일 구조)
from tensorflow.keras import Sequential
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False   # 가중치 동결 — '백만 장의 눈'을 그대로 빌림

# 2. 커스텀 분류 헤드 결합  (Rescaling: MobileNetV2 입력 스케일 [-1,1] 정규화)
model = Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    layers.Rescaling(1.0 / 127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation="sigmoid"),   # 출력 = P(정상)  ← 라벨 트랩 기억!
])
model.summary()

t_params = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
print(f"\n[체크] Trainable params = {t_params:,}  (1,281이면 통과)")
print("비학습(동결) 파라미터는 2,257,984개 — 약 224만 개의 '빌린 눈'입니다.")

---
## STEP 5 · 학습 실행 + 학습 곡선 (33p)
T4 기준 약 **1분 30초** — 첫 에폭이 50초 남짓으로 가장 오래 걸리고, 이후 에폭은 10초 안팎으로 짧아집니다. 곡선 읽는 법:
- 손실↓·정확도↑, 그리고 **훈련·검증이 나란히** 가야 정상
- 검증은 멈췄는데 훈련만 계속 오르면 → **과적합** (암기 시작)

In [ ]:
# 학습 — 5에폭 (에폭 = 전체 데이터 한 바퀴)
import time

class TimeHistory(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.time() - self.t0)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

timer = TimeHistory()
t0 = time.time()
history = model.fit(train_ds, epochs=5, validation_data=val_ds, callbacks=[timer])
total_s = time.time() - t0
print(f"\n✅ 학습 완료 — 총 {int(total_s//60)}분 {int(total_s%60)}초 "
      f"(에폭당 {min(timer.times):.0f}~{max(timer.times):.0f}초)")

In [ ]:
# 학습 곡선 (강의안 33p 교체 캡처용 — 이 그림을 저장해 두세요)
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(h["loss"], "o-", label="Training Loss"); ax[0].plot(h["val_loss"], "s--", label="Validation Loss")
ax[0].set_title("Loss vs Epoch"); ax[0].set_xlabel("Epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(h["accuracy"], "o-", label="Training Acc"); ax[1].plot(h["val_accuracy"], "s--", label="Validation Acc")
ax[1].set_title("Accuracy vs Epoch"); ax[1].set_xlabel("Epoch"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.suptitle("학습 곡선 — 훈련·검증이 나란히 가는지 확인")
plt.tight_layout(); plt.show()
print(f"최종  train acc {h['accuracy'][-1]:.3f} / val acc {h['val_accuracy'][-1]:.3f}")

---
## STEP 6 · 테스트 715장 검증 — 혼동행렬 (34p)
모델이 학습 때 **한 번도 못 본** 테스트셋으로 채점. 같은 오답이라도 값이 다릅니다:
- **놓친 불량(FN)** = 불량→정상 오판 → 고객 유출·클레임 (가장 비쌈!)
- **헛경보(FP)** = 정상→불량 오판 → 재검사 공수·현장 피로

In [ ]:
# 테스트 평가 + 혼동행렬 (강의안 34p 레이아웃과 동일 배치)
test_ds_raw = image_dataset_from_directory(test_dir, image_size=IMG, batch_size=BATCH, shuffle=False)
test_paths = test_ds_raw.file_paths
test_ds = test_ds_raw.prefetch(AUTOTUNE)

loss, acc = model.evaluate(test_ds, verbose=0)
y_prob = model.predict(test_ds, verbose=0).ravel()          # P(정상)
y_pred = (y_prob > 0.5).astype(int)                          # 1=정상, 0=불량
y_true = np.concatenate([y.numpy() for _, y in test_ds_raw])

TN = int(((y_true == 1) & (y_pred == 1)).sum())   # 정상→정상
FP = int(((y_true == 1) & (y_pred == 0)).sum())   # 정상→불량 (헛경보)
FN = int(((y_true == 0) & (y_pred == 1)).sum())   # 불량→정상 (놓친 불량!)
TP = int(((y_true == 0) & (y_pred == 0)).sum())   # 불량→불량 (검출)

print(f"테스트 정확도: {acc:.3f}  ({acc*100:.1f}%)\n")
print("              예측:정상   예측:불량   (행 합계)")
print(f"실제 정상     TN {TN:>5}    FP {FP:>5}    {TN+FP:>5}  ← 262 이어야 정상")
print(f"실제 불량     FN {FN:>5}    TP {TP:>5}    {FN+TP:>5}  ← 453 이어야 정상")
print(f"                                  총 {TN+FP+FN+TP}  ← 715")
assert TN + FP == 262 and FN + TP == 453, "행 합계가 데이터셋 구성과 다릅니다 — 라벨 방향을 확인하세요(CASE 04)"

# 오분류 이미지 직접 열어보기 — 정확도 숫자보다 이게 진짜 리스크 관리 (34p 교훈)
mis = np.where(y_true != y_pred)[0]
print(f"\n오분류 {len(mis)}건 / 715장")
if len(mis):
    show = mis[:6]
    plt.figure(figsize=(11, 2.2 * ((len(show) + 2) // 3)))
    for i, idx in enumerate(show):
        plt.subplot((len(show) + 2) // 3, 3, i + 1)
        plt.imshow(load_img(test_paths[idx]), cmap="gray"); plt.axis("off")
        gt = "불량" if y_true[idx] == 0 else "정상"
        pd_ = "불량" if y_pred[idx] == 0 else "정상"
        plt.title(f"실제 {gt} → 예측 {pd_} (P정상={y_prob[idx]:.2f})", fontsize=9,
                  color=("red" if y_true[idx] == 0 else "darkorange"))
    plt.suptitle("오분류 사례 — 반사? 경계부? 프레임 이탈? (조별 토론 질문 1)")
    plt.tight_layout(rect=(0, 0, 1, 0.90)); plt.show()

---
## STEP 7 · Grad-CAM — 모델은 어디를 봤나 (35p)
대상 층: MobileNetV2 마지막 합성곱 **`Conv_1`**.
진단 규칙: **히트맵이 배경을 보고 있으면 모델 탓이 아니라 조명·촬영 환경을 고치라는 증거.**

In [ ]:
# Grad-CAM 구현 — '불량이라고 본 근거'를 시각화
grad_model = tf.keras.models.Model(base_model.input,
                                   [base_model.get_layer("Conv_1").output, base_model.output])
dense = model.layers[-1]   # 학습된 판정 헤드 재사용

def gradcam_heatmap(img255):                       # img255: (1,224,224,3), 0~255
    x = img255 / 127.5 - 1.0
    with tf.GradientTape() as tape:
        conv_out, feats = grad_model(x)
        p_normal = dense(tf.reduce_mean(feats, axis=(1, 2)))   # GlobalAveragePooling과 동일
        target = 1.0 - p_normal[:, 0]              # P(불량) — 출력이 P(정상)이므로! (라벨 트랩)
    grads = tape.gradient(target, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv_out[0] * pooled, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return tf.image.resize(cam[..., None], IMG)[..., 0].numpy()

def show_cam(idx, ax):
    img = np.array(load_img(test_paths[idx], target_size=IMG), dtype="float32")
    hm = gradcam_heatmap(img[None, ...])
    ax.imshow(img.astype("uint8")); ax.imshow(hm, cmap="jet", alpha=0.45); ax.axis("off")
    gt = "불량" if y_true[idx] == 0 else "정상"
    pd_ = "불량" if y_pred[idx] == 0 else "정상"
    ok = "O" if y_true[idx] == y_pred[idx] else "X"
    ax.set_title(f"[{ok}] 실제 {gt}→예측 {pd_}\nP(불량)={1 - y_prob[idx]:.2f}", fontsize=9)

caught = np.where((y_true == 0) & (y_pred == 0))[0][:2]        # 제대로 잡은 불량
wrong = mis[:2] if len(mis) else np.where(y_true == 1)[0][:2]  # 오분류(없으면 정상 예시)
picks = list(caught) + list(wrong)
fig, axes = plt.subplots(1, len(picks), figsize=(3.0 * len(picks), 3.8))
for ax_, idx in zip(np.atleast_1d(axes), picks):
    show_cam(idx, ax_)
plt.suptitle("Grad-CAM — 히트맵이 결함을 정조준하는가, 배경을 보는가")
plt.tight_layout(rect=(0, 0, 1, 0.88)); plt.show()
print("가장 인상적인 히트맵 1장을 캡처해 두세요 — 조별 제출물 ③ 입니다. (강의안 37p)")

---
## STEP 8 · 조별 결과 제출 (37p)
아래 요약을 **공유시트의 우리 조 행**에 기입:
① 최종 정확도(소수점 3자리) · ② 오분류 개수/715 · ③ Grad-CAM 캡처 1장 · ④ 촬영 하드웨어 개선 제안 1가지(조명·각도·해상도 등)

In [ ]:
# 제출용 요약 + (강사용) 33p·34p·스크립트 「」 이식 값 일괄 출력
print("=" * 52)
print("[조별 제출 요약]")
print(f"① 테스트 정확도  : {acc:.3f}")
print(f"② 오분류 개수    : {len(mis)}건 / 715장")
print("③ Grad-CAM 캡처 : 위 STEP 7 그림에서 1장 캡처")
print("④ 촬영 개선 제안 : (조별 자유 기재)")
print("=" * 52)
print()
print("[강사용 — 슬라이드·스크립트 이식 값 (8/27 리허설 확정치)]")
print(f"33p  에폭당 {min(timer.times):.0f}~{max(timer.times):.0f}초 · 총 {int(total_s//60)}분 {int(total_s%60)}초"
      f" · train {h['accuracy'][-1]:.3f} / val {h['val_accuracy'][-1]:.3f}")
print(f"34p  TN {TN} / FP {FP} / FN {FN} / TP {TP} · 정확도 {acc*100:.1f}%"
      f" · 행합 {TN+FP}/{FN+TP}/715 ✓")
print(f"스크립트 「」  33p: 총 소요·최종acc → 위 값 / 34p: 정확도 {acc*100:.1f}% · FN {FN}건 · FP {FP}건")

---
## 🔧 트러블슈팅 4케이스 (38p)
| 증상 | 조치 |
|---|---|
| ① 학습이 너무 느림 | 런타임 유형 GPU 재확인 → 그래도 느리면 `epochs=2`로 축소 |
| ② 다운로드 에러 | STEP 1의 백업 셀(구글 드라이브 우회) 실행 |
| ③ 메모리 고갈(OOM) | `BATCH = 32 → 16 → 8`로 줄여 STEP 2부터 재실행 |
| ④ 혼동행렬 해석이 이상함 | 라벨 복기: **불량=0, 정상=1**, 시그모이드 **0.5 초과 = 정상** |

문의: ㈜에이비에이치 abh@abhcst.com · 본 노트북과 데이터는 교육 후에도 무기한 접근 가능합니다.